# SAM-2 spot segmentation

Segments the spots of `data/<GROUP>/images/operculum_images` with the fine-tuned (`model_weights/sam_checkpoint.pt`) and the base SAM-2.1 Hiera-B+ model, prompted by the keypoints and boxes of `data/<GROUP>/keypoints_bboxes` (step 07).

- point prompts -> `sam2_segmentations/`, `sam2_boundingboxes/`, `sam2_masks/`, `sam2_plots/`
- box prompts -> the matching `sam2_bbox_*` folders

All outputs go to `data/<GROUP>/images/sam_data_plots/`. The fine-tuned model uses lower quality cut-offs (predicted IoU 0.5, stability 0.8) than the SAM-2 defaults because it scores its own masks lower. The SAM-2 repo is `scripts/third_party/sam2`; download the base checkpoints with its `checkpoints/download_ckpts.sh`. Set `GROUP` in the second cell.

In [ ]:
import os
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "fish_ids.csv").is_file())
SAM2_REPO = ROOT / "scripts" / "third_party" / "sam2"

if not SAM2_REPO.is_dir():
    !git clone https://github.com/facebookresearch/sam2.git "{SAM2_REPO}"
%cd {SAM2_REPO}
%pip install -e . -q
%pip install supervision -q


#!bash download_ckpts.sh


In [2]:
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "fish_ids.csv").is_file())
SAM2_REPO = ROOT / "scripts" / "third_party" / "sam2"
%cd {SAM2_REPO}

GROUP = "control"
SAM_OUT = ROOT / "data" / GROUP / "images" / "sam_data_plots"

c:\Users\tala\Downloads\PhD\Kappa Guidelines\newest_work\all_data\scripts\model inference\sam2


c:\Users\tala\.conda\envs\phdcode\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [25]:
import os
print(os.getcwd())

c:\Users\tala\Downloads\PhD\Kappa Guidelines\newest_work\all_data\scripts\model inference\sam2


In [4]:
import os
import pickle

import torch
import numpy as np
import supervision as sv

from PIL import Image, ImageChops

from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
from sam2.sam2_image_predictor import SAM2ImagePredictor

In [5]:
print("CUDA available:", torch.cuda.is_available())


print("Current working directory:", os.getcwd())
path = os.getcwd()

CUDA available: True
Current working directory: c:\Users\tala\Downloads\PhD\Kappa Guidelines\newest_work\all_data\scripts\model inference\sam2


In [6]:
if torch.cuda.is_available():
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()

    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"

checkpoint_finetuned = str(ROOT / "model_weights" / "sam_checkpoint.pt")

model_cfg_finetuned = os.path.join(
    path,
    "sam2/configs/sam2.1/sam2.1_hiera_b+.yaml"
)

sam2_finetuned = build_sam2(
    model_cfg_finetuned,
    checkpoint_finetuned,
    device=device
)


checkpoint_base = os.path.join(
    path,
    "checkpoints/sam2.1_hiera_base_plus.pt"
)

model_cfg_base = os.path.join(
    path,
    "sam2/configs/sam2.1/sam2.1_hiera_b+.yaml"
)

sam2_base = build_sam2(
    model_cfg_base,
    checkpoint_base,
    device=device
)

In [37]:
segs_dir_path_v8 = str(SAM_OUT / "sam2_segmentations")
bboxes_dir_path_v8 = str(SAM_OUT / "sam2_boundingboxes")
plots_dir_path_v8 = str(SAM_OUT / "sam2_plots")
masks_dir_path_v8 = str(SAM_OUT / "sam2_masks")

segs_dir_path_bbox = str(SAM_OUT / "sam2_bbox_segmentations")
bboxes_dir_path_bbox = str(SAM_OUT / "sam2_bbox_boundingboxes")
plots_dir_path_bbox = str(SAM_OUT / "sam2_bbox_plots")
masks_dir_path_bbox = str(SAM_OUT / "sam2_bbox_masks")

os.makedirs(segs_dir_path_v8, exist_ok=True)
os.makedirs(bboxes_dir_path_v8, exist_ok=True)
os.makedirs(plots_dir_path_v8, exist_ok=True)
os.makedirs(masks_dir_path_v8, exist_ok=True)

os.makedirs(segs_dir_path_bbox, exist_ok=True)
os.makedirs(bboxes_dir_path_bbox, exist_ok=True)
os.makedirs(plots_dir_path_bbox, exist_ok=True)
os.makedirs(masks_dir_path_bbox, exist_ok=True)

In [38]:
with open(ROOT / "data" / GROUP / "keypoints_bboxes" / "keypoints.pkl", "rb") as file:
    keypoints_data = pickle.load(file)


with open(ROOT / "data" / GROUP / "keypoints_bboxes" / "bboxes.pkl", "rb") as file:
    bboxes_data = pickle.load(file)

In [39]:
reg_roi_direc_path = str(ROOT / "data" / GROUP / "images" / "operculum_images")

In [40]:
def drop_roi_masks(detections, max_area_frac=0.05):
    """
    Remove masks that are too large to be a spot (e.g. the whole ROI / operculum).

    This replaces the old "skip index 0" rule: index 0 is simply the largest
    mask, which is usually a real spot rather than the ROI.

    Args:
        detections (sv.Detections): SAM detections.
        max_area_frac (float): Masks covering more than this fraction of the image are dropped.
    """
    if len(detections) == 0:
        return detections

    image_area = detections.mask.shape[1] * detections.mask.shape[2]

    return detections[detections.area <= max_area_frac * image_area]


def save_annotated_image(image, detections, output_path):
    """
    Create and save an annotated image using SAM detections.

    Args:
        image (np.ndarray): Input RGB image.
        detections (sv.Detections): SAM detections.
        output_path (str): Path where annotated image will be saved.
    """
    mask_annotator = sv.MaskAnnotator(
        color_lookup=sv.ColorLookup.INDEX
    )

    annotated_image = image.copy()
    annotated_image = mask_annotator.annotate(
        annotated_image,
        detections=detections
    )

    annotated_image_pil = Image.fromarray(
        annotated_image.astype(np.uint8)
    )

    annotated_image_pil.save(output_path)


def save_combined_binary_mask(detections, output_path, image_size=(1080, 1080), skip_first=True):
    """
    Save a combined binary mask from SAM detections.

    Notes:
    - Index 0 is skipped because it represents the whole ROI.
    - Spot regions are black.
    - Background is white.

    Args:
        detections (sv.Detections): SAM detections.
        output_path (str): Path where mask image will be saved.
        image_size (tuple): Output mask size as (width, height).
        skip_first (bool): Skip index 0 (whole ROI). Set False for box-prompted masks.
    """
    canvas = Image.new("L", image_size, 255)

    if detections.mask is None or len(detections) == 0:
        canvas.save(output_path)
        return

    masks_int = detections.mask.astype(int)

    for i in range(int(skip_first), masks_int.shape[0]):
        spot_mask = masks_int[i]

        spot_mask = np.where(spot_mask == 1, 0, 1)

        spot_mask_pil = Image.fromarray(
            spot_mask.astype(np.uint8) * 255
        )

        canvas = ImageChops.multiply(canvas, spot_mask_pil)

    canvas.save(output_path)


def save_segmentation_polygons(detections, output_path, image_size=1080, skip_first=True):
    """
    Convert SAM masks to normalized YOLO-style segmentation polygons.

    Output format per line:
        class_id x1 y1 x2 y2 x3 y3 ...

    Notes:
    - Index 0 is skipped because it represents the whole ROI.
    - Class ID is set to 0 for all spots.

    Args:
        detections (sv.Detections): SAM detections.
        output_path (str): Path to output txt file.
        image_size (int): Image size used for coordinate normalization.
        skip_first (bool): Skip index 0 (whole ROI). Set False for box-prompted masks.
    """
    if detections.mask is None or len(detections) == 0:
        open(output_path, "w").close()
        return

    polygons = [
        sv.mask_to_polygons(mask)
        for mask in detections.mask
    ]

    with open(output_path, "w") as file:
        for i in range(int(skip_first), len(polygons)):
            if len(polygons[i]) == 0:
                continue

            polygon = polygons[i][0].reshape(-1).astype(float)

            polygon /= image_size

            line = (0, *polygon)

            file.write(
                ("%g " * len(line)).rstrip() % line + "\n"
            )


def save_bounding_boxes(detections, output_path, image_size=1080, skip_first=True):
    """
    Save normalized YOLO-style bounding boxes from SAM detections.

    Output format per line:
        class_id x_min y_min x_max y_max

    Notes:
    - Index 0 is skipped because it represents the whole ROI.
    - Class ID is set to 0 for all spots.

    Args:
        detections (sv.Detections): SAM detections.
        output_path (str): Path to output txt file.
        image_size (int): Image size used for coordinate normalization.
        skip_first (bool): Skip index 0 (whole ROI). Set False for box-prompted masks.
    """
    if len(detections) == 0:
        open(output_path, "w").close()
        return

    bboxes = detections.xyxy

    with open(output_path, "w") as file:
        for i in range(int(skip_first), bboxes.shape[0]):
            bbox = bboxes[i].astype(float)

            bbox /= image_size

            line = (0, *bbox)

            file.write(
                ("%g " * len(line)).rstrip() % line + "\n"
            )

In [41]:
amg_kwargs_finetuned = dict(
    pred_iou_thresh=0.5,
    stability_score_thresh=0.8,
    multimask_output=False
)

amg_kwargs_base = dict()

for image_file in sorted(os.listdir(reg_roi_direc_path)):

    if not image_file.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    fn = os.path.splitext(image_file)[0]

    print(f"Processing: {fn}")

    if fn not in keypoints_data:
        print(f"No keypoints found for {fn}. Skipping.")
        continue

    point_grid = keypoints_data[fn]

    mask_generator_finetuned = SAM2AutomaticMaskGenerator(
        sam2_finetuned,
        point_grids=[point_grid],
        points_per_side=None,
        **amg_kwargs_finetuned
    )

    mask_generator_base = SAM2AutomaticMaskGenerator(
        sam2_base,
        point_grids=[point_grid],
        points_per_side=None,
        **amg_kwargs_base
    )

    image_path = os.path.join(reg_roi_direc_path, image_file)

    opened_image = np.array(
        Image.open(image_path).convert("RGB")
    )

    finetuned_result = mask_generator_finetuned.generate(opened_image)

    finetuned_detections = drop_roi_masks(
        sv.Detections.from_sam(sam_result=finetuned_result)
    )

    base_result = mask_generator_base.generate(opened_image)

    base_detections = drop_roi_masks(
        sv.Detections.from_sam(sam_result=base_result)
    )

    if len(finetuned_detections) == 0:
        print(f"Warning: fine-tuned SAM2 found no masks for {fn}; saving empty outputs.")


    save_annotated_image(
        image=opened_image,
        detections=finetuned_detections,
        output_path=os.path.join(plots_dir_path_v8, f"{fn}_sft.jpg")
    )

    save_annotated_image(
        image=opened_image,
        detections=base_detections,
        output_path=os.path.join(plots_dir_path_v8, f"{fn}_sbs.jpg")
    )


    save_combined_binary_mask(
        detections=finetuned_detections,
        output_path=os.path.join(masks_dir_path_v8, f"{fn}.jpg"),
        image_size=(1080, 1080),
        skip_first=False
    )


    save_segmentation_polygons(
        detections=finetuned_detections,
        output_path=os.path.join(segs_dir_path_v8, f"{fn}.txt"),
        image_size=1080,
        skip_first=False
    )


    save_bounding_boxes(
        detections=finetuned_detections,
        output_path=os.path.join(bboxes_dir_path_v8, f"{fn}.txt"),
        image_size=1080,
        skip_first=False
    )

    print(f"{fn} - Done")

Processing: fish_1_C_L_h
fish_1_C_L_h - Done
Processing: fish_1_C_R_h
fish_1_C_R_h - Done
Processing: fish_2_C_L_h
fish_2_C_L_h - Done
Processing: fish_2_C_R_h
fish_2_C_R_h - Done
Processing: fish_3_C_L_h
fish_3_C_L_h - Done
Processing: fish_3_C_R_h
fish_3_C_R_h - Done
Processing: fish_4_C_L_h
fish_4_C_L_h - Done
Processing: fish_4_C_R_h
fish_4_C_R_h - Done
Processing: fish_5_C_L_h
fish_5_C_L_h - Done
Processing: fish_5_C_R_h
fish_5_C_R_h - Done
Processing: fish_6_C_L_h
fish_6_C_L_h - Done
Processing: fish_6_C_R_h
fish_6_C_R_h - Done
Processing: fish_8_C_L_h
fish_8_C_L_h - Done
Processing: fish_8_C_R_h
fish_8_C_R_h - Done
Processing: fish_9_C_L_h
fish_9_C_L_h - Done
Processing: fish_9_C_R_h
fish_9_C_R_h - Done


In [42]:
predictor_finetuned = SAM2ImagePredictor(sam2_finetuned)
predictor_base = SAM2ImagePredictor(sam2_base)


def predict_with_boxes(predictor, image, boxes_norm):
    """
    Run SAM2 with one bounding-box prompt per spot.

    Args:
        predictor (SAM2ImagePredictor): SAM2 image predictor.
        image (np.ndarray): Input RGB image (H, W, 3).
        boxes_norm (np.ndarray): Normalized boxes (N, 4) as x_min, y_min, x_max, y_max.

    Returns:
        sv.Detections: One mask per input box, in the same order as boxes_norm.
    """
    h, w = image.shape[:2]

    boxes_px = np.asarray(boxes_norm, dtype=np.float32) * np.array([w, h, w, h], dtype=np.float32)

    predictor.set_image(image)

    masks, scores, _ = predictor.predict(
        box=boxes_px,
        multimask_output=False
    )

    masks = masks.reshape(-1, h, w).astype(bool)
    scores = np.asarray(scores).reshape(-1)

    return sv.Detections(
        xyxy=sv.mask_to_xyxy(masks=masks),
        mask=masks,
        confidence=scores
    )

In [43]:
for image_file in sorted(os.listdir(reg_roi_direc_path)):

    if not image_file.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    fn = os.path.splitext(image_file)[0]

    print(f"Processing (bbox prompts): {fn}")

    if fn not in bboxes_data or len(bboxes_data[fn]) == 0:
        print(f"No bounding boxes found for {fn}. Skipping.")
        continue

    boxes_norm = bboxes_data[fn]

    image_path = os.path.join(reg_roi_direc_path, image_file)

    opened_image = np.array(
        Image.open(image_path).convert("RGB")
    )

    finetuned_detections = predict_with_boxes(predictor_finetuned, opened_image, boxes_norm)
    base_detections = predict_with_boxes(predictor_base, opened_image, boxes_norm)


    save_annotated_image(
        image=opened_image,
        detections=finetuned_detections,
        output_path=os.path.join(plots_dir_path_bbox, f"{fn}_sft.jpg")
    )

    save_annotated_image(
        image=opened_image,
        detections=base_detections,
        output_path=os.path.join(plots_dir_path_bbox, f"{fn}_sbs.jpg")
    )


    save_combined_binary_mask(
        detections=finetuned_detections,
        output_path=os.path.join(masks_dir_path_bbox, f"{fn}.jpg"),
        image_size=(1080, 1080),
        skip_first=False
    )


    save_segmentation_polygons(
        detections=finetuned_detections,
        output_path=os.path.join(segs_dir_path_bbox, f"{fn}.txt"),
        image_size=1080,
        skip_first=False
    )


    save_bounding_boxes(
        detections=finetuned_detections,
        output_path=os.path.join(bboxes_dir_path_bbox, f"{fn}.txt"),
        image_size=1080,
        skip_first=False
    )

    print(f"{fn} - Done")

Processing (bbox prompts): fish_1_C_L_h
fish_1_C_L_h - Done
Processing (bbox prompts): fish_1_C_R_h
fish_1_C_R_h - Done
Processing (bbox prompts): fish_2_C_L_h
fish_2_C_L_h - Done
Processing (bbox prompts): fish_2_C_R_h
fish_2_C_R_h - Done
Processing (bbox prompts): fish_3_C_L_h
fish_3_C_L_h - Done
Processing (bbox prompts): fish_3_C_R_h
fish_3_C_R_h - Done
Processing (bbox prompts): fish_4_C_L_h
fish_4_C_L_h - Done
Processing (bbox prompts): fish_4_C_R_h
fish_4_C_R_h - Done
Processing (bbox prompts): fish_5_C_L_h
fish_5_C_L_h - Done
Processing (bbox prompts): fish_5_C_R_h
fish_5_C_R_h - Done
Processing (bbox prompts): fish_6_C_L_h
fish_6_C_L_h - Done
Processing (bbox prompts): fish_6_C_R_h
fish_6_C_R_h - Done
Processing (bbox prompts): fish_8_C_L_h
fish_8_C_L_h - Done
Processing (bbox prompts): fish_8_C_R_h
fish_8_C_R_h - Done
Processing (bbox prompts): fish_9_C_L_h
fish_9_C_L_h - Done
Processing (bbox prompts): fish_9_C_R_h
fish_9_C_R_h - Done
